In [1]:
# Standard library
import json
import random
import time
from argparse import ArgumentParser

# Third-party
import pytorch_lightning as pl
import torch
from lightning_fabric.utilities import seed
from pytorch_lightning.callbacks import LearningRateMonitor
from pytorch_lightning.profilers import AdvancedProfiler

# First-party
from neural_lam import constants, utils, config
from neural_lam.weather_dataset import WeatherDataset
from neural_lam.downscaling_dataset import DownscalingDataset
from neural_lam.netCDF_dataset import NetCDFDataset
from neural_lam.models.graph_efm import GraphEFM
from neural_lam.models.graph_fm import GraphFM
from neural_lam.models.graphcast import GraphCast
from neural_lam.models.diffusion import Diffusion
from neural_lam.models.ir_sde import IR_SDE
from neural_lam.models.stochastic_interpolants import SI
from neural_lam.models.unet import UNET
from neural_lam.models.CorrDiff import CorrDiff

In [2]:
MODELS = {
    "graphcast": GraphCast,
    "graph_fm": GraphFM,
    "graph_efm": GraphEFM,
    "diffusion": Diffusion,
    "ir_sde": IR_SDE,
    "SI": SI,
    "unet": UNET,
    "CorrDiff": CorrDiff,
}

In [5]:
config_loader = config.Config.from_file('neural_lam/NZ_hist_global_std_config.yaml')

In [6]:
train_loader = torch.utils.data.DataLoader(
    NetCDFDataset(
        start_date=config_loader.dataset.train_start_date,
        end_date=config_loader.dataset.train_end_date,
        input_path=config_loader.dataset.input_path,
        input_files=config_loader.dataset.input_files,
        ground_truth_path=config_loader.dataset.ground_truth_path,
        ground_truth_files=config_loader.dataset.ground_truth_files,
        ground_truth_stats_path=config_loader.dataset.ground_truth_stats_path,
        levels=config_loader.dataset.levels,
        is_inference_dataset=False,
        normalize_ground_truth=config_loader.dataset.normalize_ground_truth,
        subset_ds=False,
        upscale_inputs=config_loader.dataset.upscale_inputs,
        static_fields_files=config_loader.dataset.static_fields_files,
        interpolation_mode=config_loader.dataset.interpolation_mode,
        coordinate_names=config_loader.dataset.coordinate_names,
        provide_coordinates=config_loader.dataset.provide_coordinates,
        provide_day_of_year=config_loader.dataset.provide_day_of_year
    ),
    batch_size=4,
    shuffle=True,
    num_workers=1,
)

input_files: ['standardized.q_500_ACCESS-CM2_1961-1980.nc', 'standardized.q_700_ACCESS-CM2_1961-1980.nc', 'standardized.q_850_ACCESS-CM2_1961-1980.nc', 'standardized.t_500_ACCESS-CM2_1961-1980.nc', 'standardized.t_700_ACCESS-CM2_1961-1980.nc', 'standardized.t_850_ACCESS-CM2_1961-1980.nc', 'standardized.u_500_ACCESS-CM2_1961-1980.nc', 'standardized.u_700_ACCESS-CM2_1961-1980.nc', 'standardized.u_850_ACCESS-CM2_1961-1980.nc', 'standardized.v_500_ACCESS-CM2_1961-1980.nc', 'standardized.v_700_ACCESS-CM2_1961-1980.nc', 'standardized.v_850_ACCESS-CM2_1961-1980.nc', 'standardized.z_500_ACCESS-CM2_1961-1980.nc', 'standardized.z_700_ACCESS-CM2_1961-1980.nc', 'standardized.z_850_ACCESS-CM2_1961-1980.nc']
Reading static fields from ['standardized.orog_NZ_no_bounds.nc']...
Providing ground truth coordinate grid...
Providing day of year encodings...


In [7]:
test_batch = None
for batch in train_loader:
    test_batch = batch
    break

In [8]:
test_batch['LQ'].shape

torch.Size([4, 20, 128, 128])

In [9]:
test_batch['LQ'].isnan().any()

tensor(False)

In [10]:
test_batch['HQ'].shape

torch.Size([4, 2, 128, 128])

In [11]:
test_batch['HQ'].isnan().any()

tensor(False)

In [12]:
config_loader = config.Config.from_file('neural_lam/NZ_hist_global_std_config.yaml')

In [13]:
train_loader = torch.utils.data.DataLoader(
    NetCDFDataset(
        start_date=config_loader.dataset.train_start_date,
        end_date=config_loader.dataset.train_end_date,
        input_path=config_loader.dataset.input_path,
        input_files=config_loader.dataset.input_files,
        ground_truth_path=config_loader.dataset.ground_truth_path,
        ground_truth_files=config_loader.dataset.ground_truth_files,
        ground_truth_stats_path=config_loader.dataset.ground_truth_stats_path,
        levels=config_loader.dataset.levels,
        is_inference_dataset=False,
        normalize_ground_truth=config_loader.dataset.normalize_ground_truth,
        subset_ds=False,
        upscale_inputs=config_loader.dataset.upscale_inputs,
        static_fields_files=config_loader.dataset.static_fields_files,
        interpolation_mode=config_loader.dataset.interpolation_mode,
        coordinate_names=config_loader.dataset.coordinate_names,
        provide_coordinates=config_loader.dataset.provide_coordinates,
        provide_day_of_year=config_loader.dataset.provide_day_of_year
    ),
    batch_size=4,
    shuffle=True,
    num_workers=1,
)

input_files: ['standardized.q_500_ACCESS-CM2_1961-1980.nc', 'standardized.q_700_ACCESS-CM2_1961-1980.nc', 'standardized.q_850_ACCESS-CM2_1961-1980.nc', 'standardized.t_500_ACCESS-CM2_1961-1980.nc', 'standardized.t_700_ACCESS-CM2_1961-1980.nc', 'standardized.t_850_ACCESS-CM2_1961-1980.nc', 'standardized.u_500_ACCESS-CM2_1961-1980.nc', 'standardized.u_700_ACCESS-CM2_1961-1980.nc', 'standardized.u_850_ACCESS-CM2_1961-1980.nc', 'standardized.v_500_ACCESS-CM2_1961-1980.nc', 'standardized.v_700_ACCESS-CM2_1961-1980.nc', 'standardized.v_850_ACCESS-CM2_1961-1980.nc', 'standardized.z_500_ACCESS-CM2_1961-1980.nc', 'standardized.z_700_ACCESS-CM2_1961-1980.nc', 'standardized.z_850_ACCESS-CM2_1961-1980.nc']
Reading static fields from ['standardized.orog_NZ_no_bounds.nc']...
Providing ground truth coordinate grid...
Providing day of year encodings...


In [14]:
test_batch = None
for batch in train_loader:
    test_batch = batch
    break

In [15]:
test_batch['LQ'].shape

torch.Size([4, 20, 128, 128])

In [16]:
test_batch['LQ'].isnan().any()

tensor(False)

In [17]:
test_batch['HQ'].shape

torch.Size([4, 2, 128, 128])

In [18]:
test_batch['HQ'].isnan().any()

tensor(False)